# TP : du modèle linéaire au réseau à une couche cachée

## 1.0 Trois classes en anneaux

600 points du plan en trois classes de même taille : un **centre**, un **petit anneau**, un **grand
anneau**. 300 pour l'entraînement, 300 pour la validation. Il s'agit de prédire la classe d'un point
à partir de ses coordonnées $(x_1, x_2)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

class_names = ["centre", "petit anneau", "grand anneau"]


def rings(m_per_class, rng):
    """Three classes at radius 0, 1.5 and 3 around the origin, with noise on the radius."""
    y = np.repeat([0, 1, 2], m_per_class)
    radius = np.abs(1.5 * y + rng.normal(0, 0.25, len(y)))
    angle = rng.uniform(0, 2 * np.pi, len(y))
    return np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]), y


rng = np.random.default_rng(0)
X_train, y_train = rings(100, rng)
X_val, y_val = rings(100, rng)
print(f"entraînement : {X_train.shape}, validation : {X_val.shape}, "
      f"points par classe : {np.bincount(y_train)}")


def add_bias(A):
    """Prepend the constant 1 that each layer reads next to its inputs."""
    return np.hstack([np.ones((len(A), 1)), A])


def accuracy(Y_hat, y):
    return float(np.mean(np.argmax(Y_hat, axis=1) == y))


def show(ax, X, y, predict=None, title=None):
    """Scatter the three classes; with `predict`, colour the plane by the predicted class."""
    if predict is not None:
        g1, g2 = np.meshgrid(np.linspace(-4.2, 4.2, 300), np.linspace(-4.2, 4.2, 300))
        classes = np.argmax(predict(np.column_stack([g1.ravel(), g2.ravel()])), axis=1)
        ax.contourf(g1, g2, classes.reshape(g1.shape), levels=[-0.5, 0.5, 1.5, 2.5],
                    colors=["tab:blue", "tab:orange", "tab:green"], alpha=0.18)
    for k, name in enumerate(class_names):
        ax.scatter(*X[y == k].T, s=12, label=name)
    ax.set_xlim(-4.2, 4.2)
    ax.set_ylim(-4.2, 4.2)
    ax.set_aspect("equal")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    if title is not None:
        ax.set_title(title, fontsize=10)


fig, ax = plt.subplots(figsize=(5.5, 5.5))
show(ax, X_train, y_train)
ax.legend(fontsize=8)
plt.show()

## 1.1 Un modèle linéaire à trois classes

Un score par classe, $z = W \bar x$ avec $\bar x = (1, x_1, x_2)$ : $W$ a une ligne par classe, et
sa première colonne porte les biais. Puis la softmax, et l'entropie croisée contre le codage
one-hot $y$ :

$$\hat y_c = \frac{e^{z_c}}{\sum_j e^{z_j}}, \qquad
  J(W) = -\frac{1}{m}\sum_{i=1}^{m} \sum_{c} y^{(i)}_c \log \hat y^{(i)}_c, \qquad
  \boxed{\ \nabla_W J = \frac{1}{m}\sum_{i=1}^{m} \big(\hat y^{(i)} - y^{(i)}\big)\, \bar x^{(i)T}\ }$$

On descend : $W \leftarrow W - \alpha \nabla_W J$.

**Q1.** Entre deux classes, la frontière est là où leurs deux scores sont égaux. Quelle est sa
forme ? La région où une classe l'emporte peut-elle être un anneau ?

**Votre réponse :**



In [ ]:
def one_hot(y, n_classes):
    # TODO: return the (m, n_classes) matrix with a single 1 per row
    ...


def softmax(Z):
    # TODO: the softmax of each row of Z; subtract the maximum of the row first, so that exp never overflows
    ...


def cross_entropy(Y_hat, Y):
    # TODO: the mean over the rows of -log(probability given to the true class)
    ...


def train_linear(X, Y, alpha=0.5, n_steps=500):
    """Gradient descent from W = 0. Returns the weights and the cost after each step."""
    A = add_bias(X)
    W = np.zeros((Y.shape[1], A.shape[1]))
    history = []
    for _ in range(n_steps):
        # TODO: one gradient step, W <- W - alpha * (1 / m) * (Y_hat - Y)^T A with Y_hat = softmax(A W^T),
        # then append the new cost to history
        ...
    return W, np.array(history)


Y_train, Y_val = one_hot(y_train, 3), one_hot(y_val, 3)
assert Y_train.shape == (300, 3) and np.all(Y_train.sum(axis=1) == 1)
probe = np.array([[1.0, 2.0, 3.0], [1000.0, 1000.0, 1000.0]])
assert np.allclose(softmax(probe).sum(axis=1), 1.0) and np.allclose(softmax(probe)[1], 1 / 3)
assert np.allclose(softmax(probe + 5.0), softmax(probe))
assert np.isclose(cross_entropy(np.full((300, 3), 1 / 3), Y_train), np.log(3))

W_linear, history_linear = train_linear(X_train, Y_train)
linear = lambda X: softmax(add_bias(X) @ W_linear.T)

reference = LogisticRegression(C=1e12, max_iter=5000).fit(X_train, y_train)
reference_cost = cross_entropy(reference.predict_proba(X_train), Y_train)
assert abs(history_linear[-1] - reference_cost) < 1e-4

print(f"entropie croisée : descente {history_linear[-1]:.4f}, sklearn {reference_cost:.4f}, "
      f"ln 3 = {np.log(3):.4f}")
print(f"justesse : entraînement {accuracy(linear(X_train), y_train):.3f}, "
      f"validation {accuracy(linear(X_val), y_val):.3f}")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
show(ax, X_train, y_train, linear, "modèle linéaire : la classe prédite en chaque point du plan")
ax.legend(fontsize=8)
plt.show()

**Q2.** Comparez l'entropie croisée de votre descente à celle de scikit-learn, puis à $\ln 3$, celle
d'un modèle qui répondrait $(1/3,\ 1/3,\ 1/3)$ partout. Qu'est-ce qui a échoué : la descente, ou le
modèle ?

**Votre réponse :**



## 1.2 Une couche cachée

Entre $x$ et la softmax, on insère une **couche cachée** de trois unités à sigmoïde,
$\sigma(z) = 1 / (1 + e^{-z})$. La couche de sortie est le modèle de 1.1, qui lit $a^{[1]}$ au lieu
de $x$ :

$$\boxed{\ a^{[1]} = \sigma\big(W^{[1]} \bar x\big), \qquad
  \hat y = \operatorname{softmax}\big(W^{[2]} \bar a^{[1]}\big)\ }$$

$W^{[1]}$ a une ligne par unité cachée, $W^{[2]}$ une ligne par classe. Chaque couche lit ses
entrées précédées de la constante $1$, $\bar x = (1, x_1, x_2)$ et
$\bar a^{[1]} = (1, a_1, a_2, a_3)$ : la première colonne de chaque matrice porte les biais.

Cette **propagation avant** s'écrit deux fois : pour un seul point, une unité après l'autre, puis
pour tous les points d'un coup, avec un produit matriciel par couche. Les deux écritures doivent
rendre les mêmes nombres.

**Q3.** Quelles sont les dimensions de $W^{[1]}$, de $W^{[2]}$, de la matrice $A^{[1]}$ des valeurs
cachées des 300 points, et de $\hat Y$ ? Combien de poids le réseau a-t-il en tout ?

**Votre réponse :**



In [ ]:
def sigmoid(z):
    # TODO: the logistic function
    ...


def forward_one(x, W1, W2):
    """Forward pass of a single point x = (x1, x2), one unit at a time."""
    a1 = np.zeros(len(W1))
    for i in range(len(W1)):
        # TODO: hidden unit i: its bias weight W1[i, 0], plus one weight per coordinate of x, then the sigmoid
        ...
    z2 = np.zeros(len(W2))
    for c in range(len(W2)):
        # TODO: score of class c: its bias weight W2[c, 0], plus one weight per hidden unit
        ...
    return a1, softmax(z2[None, :])[0]


def forward(X, W1, W2):
    """Forward pass of all the points at once. Returns the hidden values A1 and the probabilities Y_hat."""
    # TODO: A1 = sigmoid(A0 W1^T) with A0 = add_bias(X), then Y_hat = softmax(add_bias(A1) W2^T)
    ...


rng = np.random.default_rng(0)
W1_start = rng.normal(0, 1, (3, 3))
W2_start = rng.normal(0, 1, (3, 4))

A1, Y_hat = forward(X_train, W1_start, W2_start)
assert A1.shape == (300, 3) and Y_hat.shape == (300, 3)
assert np.allclose(Y_hat.sum(axis=1), 1.0)
for i in range(len(X_train)):
    a1, y_hat = forward_one(X_train[i], W1_start, W2_start)
    assert np.allclose(a1, A1[i]) and np.allclose(y_hat, Y_hat[i])

print("les deux écritures de la propagation avant coïncident sur les 300 points")
print(f"point 0 : x = {X_train[0]}, classe « {class_names[y_train[0]]} »")
print(f"  a1    = {A1[0]}")
print(f"  y_hat = {Y_hat[0]}")
print(f"poids tirés au hasard : entropie croisée {cross_entropy(Y_hat, Y_train):.4f}, "
      f"justesse {accuracy(Y_hat, y_train):.3f}")

**Q4.** La sortie affiche la classe du point 0 et les trois probabilités que lui donne ce réseau aux
poids tirés au hasard. Retrouvez à la main sa contribution à l'entropie croisée. Quand cette
contribution est-elle grande ?

**Votre réponse :**



## 1.3 La rétropropagation

La dérivée de $J$ par rapport aux 21 poids se calcule couche par couche, **en partant de la
sortie**. On note $\delta^{[l]}$ la dérivée du coût d'un point par rapport aux sommes pondérées de
la couche $l$.

À la sortie, c'est le résidu, et le gradient est celui de 1.1 avec $\bar a^{[1]}$ à la place de
$\bar x$ :

$$\delta^{[2]} = \hat y - y, \qquad
  \frac{\partial J}{\partial W^{[2]}} = \frac{1}{m}\sum_{i=1}^{m} \delta^{[2](i)}\, \bar a^{[1](i)T} .$$

L'erreur redescend vers la couche cachée par les poids de $W^{[2]}$, multipliée par la pente de la
sigmoïde, $\sigma'(z) = a(1 - a)$ :

$$\boxed{\ \delta^{[1]} = \Big(\big(\tilde W^{[2]}\big)^T \delta^{[2]}\Big) \odot a^{[1]} \odot \big(1 - a^{[1]}\big),
  \qquad
  \frac{\partial J}{\partial W^{[1]}} = \frac{1}{m}\sum_{i=1}^{m} \delta^{[1](i)}\, \bar x^{(i)T}\ }$$

$\odot$ est le produit terme à terme, et $\tilde W^{[2]}$ est $W^{[2]}$ sans sa première colonne :
la constante $1$ n'a pas de poids entrant, elle ne reçoit aucune erreur.

On vérifie ce gradient en le calculant une seconde fois, par différences finies, un poids déplacé de
$\pm h$ à la fois :

$$\frac{\partial J}{\partial w} \approx \frac{J(w + h) - J(w - h)}{2h} .$$

**Q5.** Les différences finies coûtent deux propagations avant par poids. Combien en faut-il pour
les 21 poids de ce réseau ? Et pour un million de poids, à chaque pas de la descente ?

**Votre réponse :**



In [ ]:
def backward(X, Y, W1, W2):
    """Gradients of the cost with respect to W1 and W2, by backpropagation."""
    A1, Y_hat = forward(X, W1, W2)
    # TODO: D2 = (Y_hat - Y) / m, then G2 = D2^T add_bias(A1)
    # TODO: D1 = (D2 W2~) * A1 * (1 - A1), where W2~ is W2 without its first column, then G1 = D1^T add_bias(X)
    ...
    return G1, G2


def numerical_gradient(X, Y, W1, W2, h=1e-5):
    """The same two gradients, from the definition of a derivative: one weight moved at a time."""
    gradients = []
    for W in (W1, W2):
        G = np.zeros_like(W)
        for index in np.ndindex(*W.shape):
            original = W[index]
            # TODO: G[index] = (cost with this weight at original + h, minus cost with it at original - h) / (2 h),
            # the cost being cross_entropy(forward(X, W1, W2)[1], Y); then put the weight back to original
            ...
        gradients.append(G)
    return gradients


G1, G2 = backward(X_train, Y_train, W1_start, W2_start)
N1, N2 = numerical_gradient(X_train, Y_train, W1_start, W2_start)
assert G1.shape == W1_start.shape and G2.shape == W2_start.shape
assert np.allclose(G1, N1, atol=1e-8) and np.allclose(G2, N2, atol=1e-8)

print("gradient par rapport à W1, par rétropropagation :")
print(G1)
print("gradient par rapport à W2, par rétropropagation :")
print(G2)
print()
print(f"plus grand écart avec les différences finies : "
      f"{max(np.abs(G1 - N1).max(), np.abs(G2 - N2).max()):.1e}")
print(f"propagations avant : 1 pour la rétropropagation, "
      f"{2 * (W1_start.size + W2_start.size)} pour les différences finies")

**Q6.** Comparez le plus grand écart entre les deux gradients à la taille des dérivées affichées.
Laquelle des deux méthodes garde-t-on pour entraîner, et à quoi sert l'autre ?

**Votre réponse :**



## 1.4 L'entraînement

Dix mille pas de descente sur les deux matrices à la fois, avec $\alpha = 1$, depuis les poids tirés
au hasard de 1.2 :

$$W^{[1]} \leftarrow W^{[1]} - \alpha \frac{\partial J}{\partial W^{[1]}}, \qquad
  W^{[2]} \leftarrow W^{[2]} - \alpha \frac{\partial J}{\partial W^{[2]}} .$$

Après chaque pas, on relève le coût et la justesse, à l'entraînement et en validation.

**Q7.** Une fois tous les points d'entraînement bien classés, la justesse ne peut plus monter. Le
coût peut-il encore baisser, et jusqu'où ?

**Votre réponse :**



In [ ]:
def metrics(W1, W2):
    """The network's (training cost, validation cost, training accuracy, validation accuracy)."""
    P_train, P_val = forward(X_train, W1, W2)[1], forward(X_val, W1, W2)[1]
    return (cross_entropy(P_train, Y_train), cross_entropy(P_val, Y_val),
            accuracy(P_train, y_train), accuracy(P_val, y_val))


def train(W1, W2, alpha=1.0, n_steps=10000):
    """Gradient descent on the training set. Returns the weights and the four metrics after each step."""
    history = []
    for _ in range(n_steps):
        # TODO: one backward pass on (X_train, Y_train), one step on W1 and one on W2,
        # then append metrics(W1, W2) to history
        ...
    return W1, W2, np.array(history)


start = metrics(W1_start, W2_start)
W1, W2, history = train(W1_start, W2_start)
network = lambda X: forward(X, W1, W2)[1]

assert history.shape == (10000, 4) and history[-1, 0] < start[0]
assert accuracy(network(X_train), y_train) > 0.99

linear_metrics = (cross_entropy(linear(X_train), Y_train), cross_entropy(linear(X_val), Y_val),
                  accuracy(linear(X_train), y_train), accuracy(linear(X_val), y_val))
print(f"{'':<30}{'entropie croisée':^26}{'justesse':^26}".rstrip())
print(f"{'':<30}{'entraînement':>13}{'validation':>13}{'entraînement':>13}{'validation':>13}")
for label, row in (("modèle linéaire de 1.1", linear_metrics), ("réseau, avant le premier pas", start),
                   ("réseau, après 10 000 pas", history[-1])):
    print(f"{label:<30}{row[0]:>13.4f}{row[1]:>13.4f}{row[2]:>13.3f}{row[3]:>13.3f}")

steps = np.arange(1, len(history) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].plot(steps, history[:, 0], label="entraînement")
axes[0].plot(steps, history[:, 1], label="validation")
axes[0].axhline(linear_metrics[0], color="crimson", ls="--", lw=1, label="modèle linéaire, entraînement")
axes[0].set_yscale("log")
axes[0].set_ylabel("entropie croisée $J$")
axes[1].plot(steps, history[:, 2], label="entraînement")
axes[1].plot(steps, history[:, 3], label="validation")
axes[1].axhline(1 / 3, color="crimson", ls="--", lw=1, label="réponse constante")
axes[1].set_ylabel("justesse")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("pas")
    ax.legend(fontsize=8)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.5))
show(axes[0], X_train, y_train, linear, "modèle linéaire de 1.1")
show(axes[1], X_train, y_train, network, "réseau")
axes[1].legend(fontsize=8)
plt.show()

**Q8.** Sur les courbes d'entraînement : le coût baisse-t-il à chaque pas ? La justesse aussi ? Que
fait le coût une fois que la justesse ne monte plus ?

**Votre réponse :**



**Q9.** Les courbes de validation suivent-elles celles d'entraînement ? À quoi reconnaîtrait-on un
surapprentissage sur ces courbes ?

**Votre réponse :**



**Q10.** Comparez les régions du réseau à celles du modèle linéaire. Quelle forme ont celles du
centre et du petit anneau ? Le modèle linéaire pouvait-il les produire ?

**Votre réponse :**

